# kaggloop Colab worker

GPU ワーカー。ローカルの Claude Code が `kloop.colab` で投入したジョブを
Google Drive 経由で受け取り、GPU で実行して結果を書き戻します。

**準備:** Runtime → Change runtime type → GPU を選択。
`~/.kaggle/kaggle.json` を Drive の `MyDrive/kaggloop/kaggle.json` に置くか、
下のセルでアップロードしてください。

In [ ]:
# 1. Mount Google Drive (the shared transport between laptop and Colab).
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/kaggloop'   # <- must match KLOOP_COLAB_* on your laptop
QUEUE = f'{ROOT}/queue'
RESULTS = f'{ROOT}/results'
os.makedirs(QUEUE, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
print('queue   :', QUEUE)
print('results :', RESULTS)

In [ ]:
# 2. Kaggle credentials. Either copy kaggle.json from Drive, or upload it.
import os, shutil, pathlib
os.makedirs('/root/.kaggle', exist_ok=True)
drive_token = f'{ROOT}/kaggle.json'
if os.path.exists(drive_token):
    shutil.copy(drive_token, '/root/.kaggle/kaggle.json')
else:
    from google.colab import files
    print('Drive に kaggle.json がないためアップロードしてください。')
    up = files.upload()
    for name in up:
        shutil.move(name, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!pip -q install kaggle
print('kaggle 認証を設定しました。')

In [ ]:
# 3. Get the worker code (clone the public repo, or pull if already cloned).
import os
if not os.path.isdir('/content/kaggloop'):
    !git clone --depth 1 https://github.com/qurore/kaggloop.git /content/kaggloop
else:
    !cd /content/kaggloop && git pull --ff-only
print('worker:', '/content/kaggloop/colab/worker.py')

In [ ]:
# 4. Run the worker. It polls the queue and runs jobs on the GPU until stopped.
#    Use --once to drain the current queue and exit instead of looping.
!python /content/kaggloop/colab/worker.py \
    --queue "$QUEUE" --results "$RESULTS" --data /content/kaggle_data --loop --interval 20